# Phase 10 on Google Colab

Use this notebook when running on Google Colab (T4 or A100 GPU).
Docker does not work on Colab, so this notebook installs Ollama directly
and runs the experiment pipeline without containerization.

**Runtime recommended:** GPU (T4 minimum, A100 preferred)

**Steps:**
1. Set runtime to GPU (Runtime > Change runtime type > T4 GPU)
2. Run all cells in order
3. Results will be saved to Google Drive

In [ ]:
# ── Step 1: Mount Google Drive (results will persist here) ────────────────
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/agentic-llmops-results'
import os
os.makedirs(f'{RESULTS_DIR}/raw', exist_ok=True)
os.makedirs(f'{RESULTS_DIR}/summary', exist_ok=True)
os.makedirs(f'{RESULTS_DIR}/cache', exist_ok=True)
print(f'Results will be saved to: {RESULTS_DIR}')

In [ ]:
# ── Step 2: Clone the repo ────────────────────────────────────────────────
!git clone https://github.com/ANONYMIZED/agentic-llmops.git  # replace ANONYMIZED with your anonymized artifact repo /content/agentic-llmops
%cd /content/agentic-llmops
!pip install -r requirements.txt -q

In [ ]:
# ── Step 3: Install Ollama ────────────────────────────────────────────────
# Ollama works on Colab by running as a background process
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time

# Start Ollama server in background
proc = subprocess.Popen(['ollama', 'serve'],
                        stdout=subprocess.DEVNULL,
                        stderr=subprocess.DEVNULL)
time.sleep(5)  # wait for server to start

# Verify it is running
!curl -s http://localhost:11434/api/tags | python3 -c "import json,sys; print('Ollama running, models:', [m['name'] for m in json.load(sys.stdin).get('models',[])])"

In [ ]:
# ── Step 4: Pull models (run once — takes 10-20 min depending on GPU) ─────
# Only pull what you need. Comment out models you already have.
models = [
    'llama3.1:8b',        # critic + fixer (required for all runs)
    'codellama:7b',       # Phase 10 cross-benchmark: expected benefit model
    'qwen2.5-coder:7b',   # Phase 10 cross-benchmark: expected loss model
    'deepseek-coder:6.7b', # Phase 9b middle model
    # 'starcoder2:7b',    # Phase 9b floor model — skip if short on time
]

for model in models:
    print(f'Pulling {model}...')
    !ollama pull {model}
    print(f'Done: {model}')

In [ ]:
# ── Step 5: Symlink results dir to Google Drive ───────────────────────────
!rm -rf /content/agentic-llmops/results
!ln -s {RESULTS_DIR} /content/agentic-llmops/results
print('Results symlinked to Google Drive')

In [ ]:
# ── Step 6: Run Phase 9b (genuine T=0.3 CI study for middle models) ───────
# This runs only the Phase 9b portion of the script.
# Estimated time on T4 GPU: 3-4 hours
# Use RESUME_FROM to continue after a Colab disconnect

import subprocess
RESUME_FROM = 1  # Change this if resuming after a disconnect

result = subprocess.run(
    ['bash', 'scripts/run_phase10_cross_benchmark.sh'],
    env={**os.environ, 'RESUME_FROM': str(RESUME_FROM), 'OLLAMA_HOST': 'http://localhost:11434'},
    cwd='/content/agentic-llmops'
)
print('Exit code:', result.returncode)

In [ ]:
# ── Step 7: Compute CIs from Phase 9b results ─────────────────────────────
!python3 src/compute_phase7_ci.py \
    --input results/summary/master_results_phase9b.csv \
    --output results/summary/phase9b_ci_summary.csv

import pandas as pd
ci = pd.read_csv('results/summary/phase9b_ci_summary.csv')
print(ci.to_string(index=False))

In [ ]:
# ── Step 8: Quick results check ───────────────────────────────────────────
import pandas as pd

p9b = pd.read_csv('results/summary/master_results_phase9b.csv')
summary = p9b.groupby(['run_label', 'version'])['pass_at_1_rate'].agg(['mean','std','count'])
print('Phase 9b results summary:')
print(summary)